path = " /Volumes/workspace/default/pyspark_datasets/New Data/" 



In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
orders = spark.read.format("csv")\
    .option("header" , True) \
    .option("inferSchema" , True) \
    .load("/Volumes/workspace/default/pyspark_datasets/New Data/orders_csv.csv")
 

In [0]:
users= spark.read.format("csv")\
    .option("header" , True) \
    .option("inferSchema" , True) \
    .load("/Volumes/workspace/default/pyspark_datasets/New Data/customers_csv.csv")

In [0]:
new_orders = spark.read.format("json") \
 .load("/Volumes/workspace/default/pyspark_datasets/New Data/new_orders.json") 

spark.sql("query")


In [0]:
orders.createOrReplaceTempView("orders")
users.createOrReplaceTempView("users")

#it will convert the dataframe-> into sql table (it is temporary(Temp))



In [0]:
result=spark.sql("""
                 select * 
                 from orders
                 where amount>300
                 """)
result.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
+--------+-----------+----------+------+--------+---------+----------+



In [0]:
orders.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       2|        102|      1002|   300|       1|  Pending|2024-01-02|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       4|        101|      1002|   200|       1|Cancelled|2024-01-04|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
|       6|        105|      1004|   150|       1|  Pending|2024-01-06|
+--------+-----------+----------+------+--------+---------+----------+



In [0]:
users.show()

+-----------+------+-------+-----------+
|customer_id|  name|country|signup_date|
+-----------+------+-------+-----------+
|        101| Kusum|  India| 2023-12-01|
|        102|Nikesh|  India| 2023-12-05|
|        103|  Nick|    USA| 2023-12-10|
|        104|Sujana|     UK| 2023-12-15|
|        105| Nisha| Canada| 2023-12-20|
+-----------+------+-------+-----------+



In [0]:
#Join with aggregration

revenue_df=spark.sql("""
                     select 
                     u.country,
                     count(o.order_id) as total_orders, 
                     sum(o.amount) as total_revenue
                     from orders o  
                     join users u 
                     on o.customer_id = u.customer_id
                     group By u.country
                     order By total_revenue desc
                
                     """)

In [0]:
revenue_df.show()

+-------+------------+-------------+
|country|total_orders|total_revenue|
+-------+------------+-------------+
|  India|           3|         1000|
|     UK|           1|         1000|
|    USA|           1|          700|
| Canada|           1|          150|
+-------+------------+-------------+



In [0]:
min_amount=300
filter_df=spark.sql(f"""
                    select *
                    from orders
                    where amount >{min_amount}
                    """)

In [0]:
filter_df.show()

+--------+-----------+----------+------+--------+---------+----------+
|order_id|customer_id|product_id|amount|quantity|   status|order_date|
+--------+-----------+----------+------+--------+---------+----------+
|       1|        101|      1001|   500|       2|Completed|2024-01-01|
|       3|        103|      1003|   700|       3|Completed|2024-01-03|
|       5|        104|      1001|  1000|       4|Completed|2024-01-05|
+--------+-----------+----------+------+--------+---------+----------+



In [0]:
#write spark.sql queries with the help of window function

In [0]:
new_orders.createOrReplaceTempView("new_orders")


In [0]:
new_orders.printSchema()

root
 |-- amount: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- status: string (nullable = true)



In [0]:
new_orders.show()

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   750|        103|       3|      1003|       3|Completed|
|   400|        106|       7|      1005|       2|Completed|
+------+-----------+--------+----------+--------+---------+



In [0]:
#we have to update and insert
#find new records with the help of join

new_orders=spark.sql("""
                     select  n.*
                     from new_orders n
                     left join orders o 
                     on n.order_id = o.order_id
                     where o.order_id is null
                
                     """)


In [0]:
new_orders.show()

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   400|        106|       7|      1005|       2|Completed|
+------+-----------+--------+----------+--------+---------+



In [0]:
#updated record

updated_record=spark.sql("""
                         select n.*
                         from new_orders n
                         join orders o 
                          on n.order_id = o.order_id
                        
                         """)


In [0]:
updated_record.show()

#CDC(change data capture)
#Delta table

+------+-----------+--------+----------+--------+---------+
|amount|customer_id|order_id|product_id|quantity|   status|
+------+-----------+--------+----------+--------+---------+
|   750|        103|       3|      1003|       3|Completed|
+------+-----------+--------+----------+--------+---------+

